In [1]:
from pathlib import Path
import json
import re
import uuid
import torch
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
import pandas as pd
import random
import os
import accelerate
from unsloth import FastLanguageModel, get_chat_template
import gc

c:\PokeAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1120 01:00:31.223000 8044 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\PokeAI\.venv\Lib\site-packages\unsloth\__init__.py:1554: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:

# Função de busca no Qdrant
def recuperar_contexto(pergunta, k=5):
    embedding = modelo.encode(pergunta, normalize_embeddings=True).tolist()
    resultados = client.query_points(collection_name=COLECAO, query=embedding, limit=k, with_payload=True)
    resultados = resultados.points
    if not resultados:
        print("Nenhum resultado encontrado.")
        return []
    print("=" * 90)
    print(f"PERGUNTA: {pergunta}")
    print("=" * 90)
    contextos = []
    for indice, resultado in enumerate(resultados, start=1):
        payload = resultado.payload
        '''print(f"\n--- Resultado {indice} ---")
        print(f"Arquivo: {payload.get('arquivo')}")
        print(f"Título: {payload.get('titulo')}")
        print(f"Tipo: {payload.get('tipo')}")
        print(f"Score vetorial: {resultado.score:.5f}")
        print("\nContexto recuperado:")
        print(payload.get("texto", ""))'''
        contextos.append({
            "arquivo": payload.get("arquivo"),
            "texto": payload.get("texto", "")
        })
    #client.close()
    return contextos

def montar_contexto(contextos):
    if not contextos:
        return "Nenhum contexto relevante foi recuperado pelo sistema RAG."

    partes = []

    for indice, contexto in enumerate(contextos, start=1):
        trecho = f"""
<Fonte numero="{indice}">
Arquivo: {contexto['arquivo']}
Conteúdo:
{contexto['texto']}
</Fonte>
"""
        partes.append(trecho)

    return "\n".join(partes)
# Carregamento do modelo

os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

model_name = 'Llama-3.2-3B-Instruct'
path = f'C:/PokeAI/Models/{model_name}'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=path,
    max_seq_length=4096,
    load_in_4bit=True,
    device_map="cuda:0",
)

model.generation_config.max_length = None
FastLanguageModel.for_inference(model)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Configurações Qdrant
PASTA_QDRANT = "C:/PokeAI/databases/RAG/Metagame/qdrant_db"
COLECAO = "pokeai_minilm_text"
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Quantidade de trechos para recuperar
K  = 5

modelo = SentenceTransformer(MODELO, device="cuda")
client = QdrantClient(path=PASTA_QDRANT)

# Carregamento do arquivo CSV com as perguntas
df = pd.read_csv("C:/PokeAI/databases/P&R/Metagame/02112026_Perguntas_Metagame.csv")

if "contexto_RAG" not in df.columns:
    df["contexto_RAG"] = ""

if "resposta_gerada" not in df.columns:
    df["resposta_gerada"] = ""

# Prompt
instruction = open("C:/PokeAI/competitive-poke-ai/Codes/Prompts/prompt_Inferência.txt", "r", encoding="utf-8").read()

# Iterar sobre todas as perguntas fazendo a inferência
for index, row in df.iterrows():
    print(f"Avaliando questão {index+1}")
    pergunta = row['pergunta']
    contextos = recuperar_contexto(pergunta, k=K)
    trechos_RAG = montar_contexto(contextos)
    
    infos = f'''
        Contexto recuperado pelo RAG:

        <contexto>
        {trechos_RAG}
        </contexto>

        Pergunta do usuário:

        <pergunta>
        {pergunta}
        </pergunta>

        Responda seguindo rigorosamente as instruções acima.
    '''

    prompt = instruction + infos
    messages = [{"role": "user", "content": prompt}]

    if model_name == 'Qwen3.5-4B':
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    else:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    #inputs = tokenizer([prompt], return_tensors="pt", padding=True).to("cuda")
    inputs = tokenizer(text=[prompt], return_tensors="pt", padding=True).to("cuda")
    prompt_len = inputs.input_ids.shape[1]

    outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, do_sample=False)#, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)

    generated_tokens = outputs[0][prompt_len:]
    hypothesis = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    print(f"\n\nResposta COMPLETA: {hypothesis}\n{'-'*80}")
    if "</think>" in hypothesis:
        hypothesis = hypothesis.split("</think>", 1)[1].strip()

    df.at[index, 'contexto_RAG'] = trechos_RAG
    df.at[index, 'resposta_gerada'] = hypothesis
    print(hypothesis)
    break
df.to_csv("C:/PokeAI/databases/P&R/Metagame/Results/Eval_Perguntas_Metagame.csv", index=False)
client.close()


==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce GTX 1660 SUPER. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 254/254 [00:54<00:00,  4.69it/s]
Unsloth: Will load C:/PokeAI/Models/Llama-3.2-3B-Instruct as a legacy tokenizer.


Unsloth: C:/PokeAI/Models/Llama-3.2-3B-Instruct has no pad_token. Using pad_token = <|finetune_right_pad_id|>.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 651.89it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Avaliando questão 1


KeyError: 'Pergunta'

In [3]:
# Iterar sobre todas as perguntas fazendo a inferência
for index, row in df.iterrows():
    print(f"Avaliando questão {index+1}")
    pergunta = row['pergunta']
    contextos = recuperar_contexto(pergunta, k=K)
    trechos_RAG = montar_contexto(contextos)
    
    infos = f'''
        Contexto recuperado pelo RAG:

        <contexto>
        {trechos_RAG}
        </contexto>

        Pergunta do usuário:

        <pergunta>
        {pergunta}
        </pergunta>

        Responda seguindo rigorosamente as instruções acima.
    '''

    prompt = instruction + infos
    messages = [{"role": "user", "content": prompt}]

    if model_name == 'Qwen3.5-4B':
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    else:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    #inputs = tokenizer([prompt], return_tensors="pt", padding=True).to("cuda")
    inputs = tokenizer(text=[prompt], return_tensors="pt", padding=True).to("cuda")
    prompt_len = inputs.input_ids.shape[1]

    outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, do_sample=False)#, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)

    generated_tokens = outputs[0][prompt_len:]
    hypothesis = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    print(f"\n\nResposta COMPLETA: {hypothesis}\n{'-'*80}")
    if "</think>" in hypothesis:
        hypothesis = hypothesis.split("</think>", 1)[1].strip()

    df.at[index, 'contexto_RAG'] = trechos_RAG
    df.at[index, 'resposta_gerada'] = hypothesis
    print(hypothesis)
    break
df.to_csv("C:/PokeAI/databases/P&R/Metagame/Results/02112026_Perguntas_Metagame.csv", index=False)

Avaliando questão 1
PERGUNTA: Qual é o Pokémon com maior taxa de uso absoluto no metagame VGC 2026 Regulation M-B em alto elo (1760+ Elo, Bo3)?

--- Resultado 1 ---
Arquivo: 18_pokemon_geral_regmb_b03.md
Título: Pokémon: Incineroar - Ranking Geral
Tipo: markdown
Score vetorial: 0.72708

Contexto recuperado:
# Pokémon: Incineroar - Ranking Geral
No ranking geral de utilização do metagame competitivo de Pokémon, o Pokémon Incineroar ocupa a 4ª posição (Rank 4). Ele registra uma taxa de uso ponderada (usage percent) de 42.30358% em um universo de 179,434 batalhas analisadas. Em números brutos de aparições (raw count), Incineroar esteve presente em 115,366 batalhas, o que equivale a uma porcentagem bruta de presença de 32.147%. Ao ajustar pelas batalhas de alto nível (real count), ele contabilizou 65,976 aparições, representando uma porcentagem real de 39.531%. O peso médio das equipes que contêm este Pokémon (avg weight team) é de 0.071.

--- Resultado 2 ---
Arquivo: 70_pokemon_geral_regm

In [9]:
contextos = recuperar_contexto("Qual é a taxa de uso e a principal função do Incineroar em partidas Bo3 de alto elo?", k=K)
contextos

PERGUNTA: Qual é a taxa de uso e a principal função do Incineroar em partidas Bo3 de alto elo?

--- Resultado 1 ---
Arquivo: 40_regmb_Incineroar _ Champions _ Smogon Strategy Pokedex.md
Título: Incineroar — Competitive Strategies
Tipo: markdown
Score vetorial: 0.46394

Contexto recuperado:
# Incineroar — Competitive Strategies

## Identification and formats
- Pokémon: 
Incineroar
- Formats: 
VGC26 Regulation M-A / VGC26 Regulation M-B
- Competitive Role: 
Pivot Support
- Nature: 
Impish
- Level: 
50

## Core strategy and moves
In other gens: SM / SS / SV / Champions

Incineroar

HP: 95 Attack: 115 Defense: 90 Sp. Atk: 80 Sp. Def: 90 Speed: 60

Strategies (  en)

Formats: VGC26 Regulation M-A / VGC26 Regulation M-B

Written by pengu

Pivot Support

Move 1:

Move 2:

Move 3:

Move 4:

Item:

Ability:

Nature: Impish / Careful

Level: 50

SPs: 32 HP / 11 SpD / 23 Spe 32 HP / 23 Def / 11 SpD

Incineroar is able to use its stellar supportive moves and ability, strong defensive typing, and s

[{'arquivo': '40_regmb_Incineroar _ Champions _ Smogon Strategy Pokedex.md',
  'texto': "# Incineroar — Competitive Strategies\n\n## Identification and formats\n- Pokémon: \nIncineroar\n- Formats: \nVGC26 Regulation M-A / VGC26 Regulation M-B\n- Competitive Role: \nPivot Support\n- Nature: \nImpish\n- Level: \n50\n\n## Core strategy and moves\nIn other gens: SM / SS / SV / Champions\n\nIncineroar\n\nHP: 95 Attack: 115 Defense: 90 Sp. Atk: 80 Sp. Def: 90 Speed: 60\n\nStrategies (  en)\n\nFormats: VGC26 Regulation M-A / VGC26 Regulation M-B\n\nWritten by pengu\n\nPivot Support\n\nMove 1:\n\nMove 2:\n\nMove 3:\n\nMove 4:\n\nItem:\n\nAbility:\n\nNature: Impish / Careful\n\nLevel: 50\n\nSPs: 32 HP / 11 SpD / 23 Spe 32 HP / 23 Def / 11 SpD\n\nIncineroar is able to use its stellar supportive moves and ability, strong defensive typing, and solid stats to be one of the best Pokemon in Regulation M-B, tanking attacks from the likes of Kingambit and Mega Charizard Y while shutting down and even o